# 실습 2회: HTML 파싱과 감사보고서 구조 분석

서울대학교 핀테크 실습 과정 | 12기

---

## 학습 목표

1. HTML 문서의 DOM(Document Object Model) 구조와 태그 체계를 이해한다.
2. `BeautifulSoup` 라이브러리를 활용하여 HTML을 파싱하고 탐색한다.
3. EUC-KR 인코딩으로 작성된 감사보고서 파일을 안정적으로 로드한다.
4. CSS 클래스 기반의 섹션 계층(SECTION-1/2) 구조를 분석하여 각 섹션의 텍스트를 추출한다.
5. 추출 결과를 구조화된 JSON 형식으로 저장하여 다음 실습을 위한 입력 데이터를 준비한다.

---

## 실습 데이터

| 항목 | 내용 |
|:---|:---|
| 파일명 | `감사보고서_2024.htm` |
| 파일 인코딩 | EUC-KR |
| 문서 분량 | 약 11,400줄 |
| 포함 섹션 | 독립된 감사인의 감사보고서, 재무제표, 주석, 내부회계관리제도, 외부감사 실시내용 |

---

## 전체 처리 파이프라인

```
[1] 패키지 설치 및 파일 업로드
 |
[2] EUC-KR 인코딩 감지 및 파일 로드
 |
[3] BeautifulSoup 파싱 (DOM 트리 구성)
 |
[4] 섹션 헤더 탐색 (h2/h3 + SECTION-1/2 클래스)
 |
[5] 섹션별 텍스트 수집 (표 제외)
 |
[6] JSON 저장
```

---

## Step 1. 패키지 설치

Google Colab을 포함한 모든 환경에서 실행 가능하도록
필요한 라이브러리를 먼저 설치한다.

| 라이브러리 | 용도 |
|:---|:---|
| `beautifulsoup4` | HTML/XML 파싱의 핵심 라이브러리 |
| `lxml` | C 기반 고성능 HTML 파서 (BeautifulSoup 백엔드) |
| `html5lib` | HTML5 표준 준수 파서 (비정형 HTML 대응) |

In [31]:
# Colab 및 로컬 환경 공통 패키지 설치
#!pip install beautifulsoup4 lxml html5lib --quiet
print("패키지 설치 완료")

패키지 설치 완료


In [32]:
# 설치된 라이브러리 버전 확인
import importlib

for pkg in ("bs4", "lxml", "html5lib"):
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "(버전 정보 없음)")
        print(f"  {pkg:<12} {ver}")
    except ImportError:
        print(f"  {pkg:<12} 설치되지 않음")

  bs4          4.14.3
  lxml         6.0.2
  html5lib     1.1


---

## Step 2. 실습 파일 준비

### Google Colab 사용 시 파일 업로드 방법

두 가지 방법 중 하나를 선택하여 사용한다.

**방법 A: 직접 업로드 (간단)**
```python
from google.colab import files
uploaded = files.upload()  # 파일 선택 창이 열림
# '감사보고서_2024.htm' 파일을 선택하면 /content/ 폴더에 저장됨
```

**방법 B: Google Drive 연동 (권장)**
```python
from google.colab import drive
drive.mount('/content/drive')
# 이후 파일 경로: '/content/drive/MyDrive/실습데이터셋/감사보고서_2024.htm'
```

### 로컬(Jupyter) 사용 시
파일 경로를 노트북 위치 기준 상대 경로로 설정한다.

In [33]:
import os
from pathlib import Path

# ── 환경 자동 감지 ─────────────────────────────────────────────────────
IN_COLAB = "google.colab" in str(globals().get("__builtins__", ""))
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"실행 환경: {'Google Colab' if IN_COLAB else '로컬 Jupyter'}")

# ── 파일 경로 설정 ──────────────────────────────────────────────────────
if IN_COLAB:
    # [Colab] 아래 두 방법 중 하나를 주석 해제하여 사용

    # 방법 A: 직접 업로드
    # from google.colab import files
    # uploaded = files.upload()
    # HTML_PATH = Path(list(uploaded.keys())[0])

    # 방법 B: Google Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # HTML_PATH = Path('/content/drive/MyDrive/실습데이터셋/감사보고서_2024.htm')

    # 기본값 (방법 A 사용 후 파일명 직접 지정)
    HTML_PATH = Path("/content/감사보고서_2024.htm")
else:
    # [로컬] 노트북 기준 상대 경로
    HTML_PATH = Path("../실습데이터셋/감사보고서_2024.htm")

# ── 파일 존재 여부 확인 ─────────────────────────────────────────────────
if HTML_PATH.exists():
    size_kb = HTML_PATH.stat().st_size / 1024
    print(f"파일 확인: {HTML_PATH.name}  ({size_kb:.0f} KB)")
else:
    print(f"파일을 찾을 수 없습니다: {HTML_PATH}")
    print("위 셀의 주석을 해제하여 파일을 업로드하거나 경로를 수정하세요.")

실행 환경: 로컬 Jupyter
파일 확인: 감사보고서_2024.htm  (521 KB)


---

## Step 3. 기본 임포트

이후 실습 전체에서 사용할 표준 라이브러리와 BeautifulSoup을 임포트한다.

| 모듈 | 용도 |
|:---|:---|
| `re` | 정규표현식(Regular Expression): 텍스트 패턴 검색·추출 |
| `json` | JSON 파일 읽기·쓰기 |
| `pathlib.Path` | 파일 경로를 객체로 다루는 표준 라이브러리 |
| `bs4.BeautifulSoup` | HTML 파싱 및 DOM 탐색 |
| `bs4.Tag` | BeautifulSoup의 HTML 태그 노드 클래스 |
| `bs4.NavigableString` | BeautifulSoup의 텍스트 노드 클래스 |

**정규표현식(regex)** 이란 문자열에서 특정 패턴을 찾거나 추출하는 규칙이다.
예: `re.findall(r'\d+', "주석 4번")` → `['4']` (숫자만 추출)

In [34]:
import re
import json
from typing import List, Dict, Any, Optional

from bs4 import BeautifulSoup, Tag, NavigableString

print("임포트 완료")

임포트 완료


---

## Step 4. HTML 문서의 구조: 태그, 속성, DOM

### 4.1 HTML이란 무엇인가

HTML(HyperText Markup Language)은 웹 페이지의 내용과 구조를 기술하는 텍스트 언어이다.
우리가 Chrome이나 Safari에서 보는 모든 웹 페이지는 HTML로 작성되어 있다.
DART에서 내려받는 감사보고서 `.htm` 파일도 메모장으로 열면 읽을 수 있는 텍스트 파일이며,
그 안에 HTML 문법으로 섹션, 표, 문단의 구조가 정의되어 있다.

---

### 4.2 태그(Tag): HTML의 기본 단위

HTML의 모든 구성 요소는 **태그(tag)**로 표현된다.
태그는 꺾쇠 괄호 `< >`로 감싼 이름이며, 보통 **여는 태그**와 **닫는 태그**가 쌍을 이룬다.

```
형식:  <태그이름 속성1="값" 속성2="값"> 내용 </태그이름>
                ↑                         ↑
          여는 태그                  닫는 태그 (/ 가 붙음)
```

실제 예시:
```html
<h2 class="SECTION-1" id="toc_2">독립된 감사인의 감사보고서</h2>
<p>본 감사인은 재무제표를 감사하였습니다.</p>
<table>
  <tr>
    <th>과목</th>
    <td>현금및현금성자산</td>
  </tr>
</table>
```

**감사보고서에서 자주 등장하는 태그:**

| 태그 | 이름 | 역할 |
|:---|:---|:---|
| `<h1>` ~ `<h6>` | Heading | 제목. 숫자가 클수록 하위 레벨 제목 |
| `<p>` | Paragraph | 문단 텍스트. 가장 많이 사용됨 |
| `<div>` | Division | 구역 구분. 시각적 구조 용도 |
| `<table>` | Table | 표의 컨테이너 |
| `<tr>` | Table Row | 표의 한 행 |
| `<th>` | Table Header | 표의 헤더 셀 (볼드체) |
| `<td>` | Table Data | 표의 일반 데이터 셀 |
| `<br>` | Line Break | 줄바꿈 (닫는 태그 없음) |

---

### 4.3 속성(Attribute): 태그에 붙는 추가 정보

태그의 여는 부분 안에 `속성명="값"` 형태로 추가 정보를 지정한다.

```html
<h2   class="SECTION-1"   id="toc_2">독립된 감사인의 감사보고서</h2>
       ↑                    ↑
  CSS 클래스 지정        고유 식별자 지정
```

**`class` 속성**이란?
같은 스타일이나 역할을 가진 태그들을 묶는 레이블이다.
여러 태그가 동일한 class를 공유할 수 있다.
DART 감사보고서에서는 `SECTION-1`, `SECTION-2` 등의 class로 섹션의 계층 레벨을 표시한다.

**`id` 속성**이란?
문서 안에서 해당 태그를 유일하게 식별하는 이름이다.
문서 전체에서 동일한 id는 하나의 태그만 가질 수 있다.
감사보고서에서는 `toc_2`, `toc_3` 등으로 목차 위치를 표시한다.

**CSS 클래스와 스타일 연결**: 문서 상단의 `<style>` 태그에서 각 클래스의 글꼴, 색상, 크기 등이 정의된다.
```css
P.SECTION-1 { font-size: 18pt; color: #035fa3; font-weight: bold; }
P.SECTION-2 { font-size: 16pt; color: #038ed5; }
```
이 덕분에 브라우저는 `SECTION-1` 클래스의 텍스트를 파란색 큰 글씨로 표시한다.

---

### 4.4 DOM(Document Object Model): 트리로 보는 HTML

브라우저나 파이썬이 HTML을 해석할 때, 태그들의 **포함 관계**를 트리(tree) 구조로 변환한다.
이 트리를 **DOM(Document Object Model)**이라 한다.

```
<html>                                      ← 루트(최상위) 노드
  └── <body>
        ├── <h2 class="SECTION-1">감사 의견  ← Tag 노드
        │
        ├── <p>본 감사인은 감사하였습니다.</p> ← Tag 노드
        │     └── "본 감사인은 감사하였습니다." ← 텍스트 노드(NavigableString)
        │
        └── <table>
              └── <tr>
                    ├── <th>과목</th>
                    └── <td>현금</td>
```

DOM 트리에서 노드 간의 관계를 정확히 이해해야 탐색 코드를 작성할 수 있다.

| 관계 | 설명 | 예시 |
|:---|:---|:---|
| **부모(parent)** | 나를 직접 감싸는 상위 노드 | `<p>`의 부모 → `<body>` |
| **자식(children)** | 내가 직접 감싸는 하위 노드들 | `<tr>`의 자식 → `<th>`, `<td>` |
| **형제(siblings)** | 같은 부모를 공유하는 노드들 | 두 번째 `<p>`의 형제 → 첫 번째 `<p>` |
| **조상(ancestors)** | 부모의 부모... 모든 상위 노드 | `<td>`의 조상 → `<tr>`, `<table>`, `<body>` |
| **자손(descendants)** | 자식의 자식... 모든 하위 노드 | `<table>`의 자손 → `<tr>`, `<th>`, `<td>` |

이 관계들은 코드에서 `tag.parent`, `tag.children`, `tag.next_siblings`, `tag.descendants` 로 접근한다.

---

### 4.5 BeautifulSoup: 파이썬에서 DOM을 다루는 라이브러리

`BeautifulSoup`은 HTML 텍스트를 파이썬 객체 트리로 변환해 주는 라이브러리이다.
변환 후에는 파이썬 코드로 태그를 검색하고 텍스트를 추출할 수 있다.

```python
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_text, "html.parser")  # HTML 파싱

# 1) 태그 검색
soup.find("h2")                     # 첫 번째 h2 태그 반환
soup.find_all("p")                  # 모든 p 태그를 리스트로 반환
soup.find(id="toc_2")               # id="toc_2" 인 태그 반환
soup.find_all(class_="SECTION-1")   # class="SECTION-1" 인 모든 태그

# 2) 속성 및 텍스트 추출
tag.get("class")                    # 클래스 속성값 반환 (리스트)
tag.get("id")                       # id 속성값 반환
tag.get_text(strip=True)            # 태그 내부 텍스트 (공백 제거)
tag.get_text(" ", strip=True)       # 태그 내부 텍스트 (공백으로 연결)

# 3) 트리 탐색
tag.parent                          # 부모 노드
tag.next_siblings                   # 다음 형제 노드들 (이터레이터)
tag.find_all("td")                  # 하위에서 모든 td 탐색
```

BeautifulSoup의 두 핵심 노드 타입:
- **`Tag`**: `<h2>`, `<p>`, `<table>` 등 HTML 태그 노드
- **`NavigableString`**: `"본 감사인은..."` 처럼 태그가 아닌 순수 텍스트 노드

---

### 실제 감사보고서의 최상위 섹션 구조

```html
<h2 class="SECTION-1" id="toc_2">독립된 감사인의 감사보고서</h2>
<h2 class="SECTION-1" id="toc_3">(첨부) 재무제표</h2>
<h3 class="SECTION-2" id="toc_4">주석</h3>
<h2 class="SECTION-1" id="toc_5">내부회계관리제도</h2>
<h2 class="SECTION-1" id="toc_6">외부감사 실시내용</h2>
```

이 문서에서는 `h2` / `h3` 태그의 `class` 속성으로 섹션의 계층을 구분한다.
즉, **CSS 클래스값(SECTION-1/2)**이 섹션 레벨을 결정하는 실질적 기준이다.

In [35]:
# [실습 4-1] 간단한 HTML 예제로 DOM 탐색 체험
sample_html = """
<html>
<body>
  <h2 class="SECTION-1" id="toc_2">독립된 감사인의 감사보고서</h2>
  <p>본 감사인은 재무제표를 감사하였습니다.</p>
  <p>감사 결과 <b>적정의견</b>을 표명합니다.</p>
  <h2 class="SECTION-1" id="toc_3">(첨부) 재무제표</h2>
  <p>(단위: 백만원)</p>
  <table>
    <tr><th>과목</th><th>주석</th><th>당기</th><th>전기</th></tr>
    <tr><td>현금</td><td>4</td><td>50,000</td><td>45,000</td></tr>
    <tr><td>매출채권</td><td>5</td><td>120,000</td><td>110,000</td></tr>
  </table>
</body>
</html>
"""

soup_ex = BeautifulSoup(sample_html, "html.parser")

# 1) h2 태그 전체 탐색
print("[h2 태그 목록]")
for h2 in soup_ex.find_all("h2"):
    print(f"  id={h2.get('id')!r:12}  class={h2.get('class')}  text={h2.get_text(strip=True)!r}")

# 2) class 속성으로 탐색
print()
print("[class='SECTION-1' 인 태그]")
for tag in soup_ex.find_all(class_="SECTION-1"):
    print(f"  <{tag.name}> id={tag.get('id')!r}")

# 3) id로 특정 태그 선택
print()
toc3 = soup_ex.find(id="toc_3")
print(f"[find(id='toc_3')] → {toc3}")

[h2 태그 목록]
  id='toc_2'       class=['SECTION-1']  text='독립된 감사인의 감사보고서'
  id='toc_3'       class=['SECTION-1']  text='(첨부) 재무제표'

[class='SECTION-1' 인 태그]
  <h2> id='toc_2'
  <h2> id='toc_3'

[find(id='toc_3')] → <h2 class="SECTION-1" id="toc_3">(첨부) 재무제표</h2>


In [36]:
# [실습 4-2] next_siblings: 형제 노드 순회
# h2 태그 이후에 어떤 노드들이 나오는지 확인한다

first_h2 = soup_ex.find("h2")
print(f"시작 태그: {first_h2}")
print()
print("[next_siblings 목록]")
for sib in first_h2.next_siblings:
    if isinstance(sib, Tag):
        print(f"  Tag       <{sib.name}>  text={sib.get_text(strip=True)[:40]!r}")
    elif isinstance(sib, NavigableString):
        text = str(sib).strip()
        if text:
            print(f"  NavString {text!r}")

시작 태그: <h2 class="SECTION-1" id="toc_2">독립된 감사인의 감사보고서</h2>

[next_siblings 목록]
  Tag       <p>  text='본 감사인은 재무제표를 감사하였습니다.'
  Tag       <p>  text='감사 결과적정의견을 표명합니다.'
  Tag       <h2>  text='(첨부) 재무제표'
  Tag       <p>  text='(단위: 백만원)'
  Tag       <table>  text='과목주석당기전기현금450,00045,000매출채권5120,000110,0'


In [37]:
# [실습 4-3] 테이블 데이터 탐색: tr → td/th

table = soup_ex.find("table")
print("[테이블 행별 셀 내용]")
for tr in table.find_all("tr"):
    cells = [cell.get_text(strip=True) for cell in tr.find_all(["td", "th"])]
    print(f"  {cells}")

# 특정 셀 접근: 첫 번째 데이터 행의 당기 금액
data_rows = table.find_all("tr")[1:]  # 헤더 제외
first_row_cells = data_rows[0].find_all(["td", "th"])
print()
print(f"첫 번째 데이터 행 당기 금액: {first_row_cells[2].get_text(strip=True)!r}")

[테이블 행별 셀 내용]
  ['과목', '주석', '당기', '전기']
  ['현금', '4', '50,000', '45,000']
  ['매출채권', '5', '120,000', '110,000']

첫 번째 데이터 행 당기 금액: '50,000'


---

## Step 5. 감사보고서 HTML 로드

### 5.1 인코딩(Encoding)이란 무엇인가

컴퓨터는 모든 데이터를 **0과 1의 바이트(byte)**로 저장한다.
텍스트 파일을 저장할 때는 "어떤 문자를 어떤 숫자로 표현할 것인가"를 정의한 **인코딩 규칙**이 필요하다.

```
'A'  → ASCII: 65번 (0x41)
'가' → UTF-8: 3바이트 (0xEA 0xB0 0x80)
'가' → EUC-KR: 2바이트 (0xB0 0xA1)
```

같은 바이트 열이라도 어떤 인코딩 규칙으로 해석하느냐에 따라 완전히 다른 문자가 나온다.
인코딩이 맞지 않으면 한글이 `???` 또는 `â°í` 처럼 깨진 문자(mojibake)로 보인다.

**주요 인코딩 방식:**

| 인코딩 | 설명 | 특징 |
|:---|:---|:---|
| **ASCII** | 영문자·숫자·기호 128자 | 1바이트, 한글 표현 불가 |
| **UTF-8** | 유니코드를 가변 길이로 인코딩 | 전 세계 모든 문자 지원, 현재 웹 표준 |
| **EUC-KR** | 한국어 완성형 표준 (KS X 1001) | 2바이트, DART 등 구형 한국 문서에 사용 |
| **CP949** | EUC-KR의 마이크로소프트 확장판 | EUC-KR 상위 호환, Windows 한국어 기본 |

---

### 5.2 DART 감사보고서의 인코딩 특성

DART(금융감독원 전자공시시스템)에서 제공하는 HTML 파일은
**EUC-KR** 또는 **CP949** 인코딩으로 작성된 경우가 많다.
파일 상단에 다음과 같이 인코딩 정보가 명시된다.

```html
<META http-equiv="Content-Type" content="text/html; charset=euc-kr">
```

파이썬에서 `open(path, encoding="utf-8")` 으로 이런 파일을 읽으면
두 가지 결과 중 하나가 발생한다.

1. **`UnicodeDecodeError`**: EUC-KR 바이트 열을 UTF-8로 해석 불가능한 경우 즉시 오류 발생
2. **글자 깨짐(mojibake)**: 해석은 성공했지만 `â°` 같은 이상한 문자가 출력되는 경우

이를 방지하기 위해 여러 인코딩을 **순서대로 시도**하는 방어적(defensive) 로드 전략을 사용한다.

```
시도 순서:  euc-kr  →  cp949  →  utf-8  →  utf-8 (손실 허용)
```

In [38]:
# [실습 5-1] 인코딩 차이 직접 확인
# 같은 바이트열을 두 인코딩으로 해석하면 어떻게 다른지 확인한다

# '재무상태표' 를 EUC-KR로 인코딩한 바이트
sample_bytes = '재무상태표'.encode('euc-kr')
print(f"원본 문자열    : '재무상태표'")
print(f"EUC-KR 바이트  : {sample_bytes}")
print()
print(f"EUC-KR 디코딩  : {sample_bytes.decode('euc-kr')!r}")
try:
    print(f"UTF-8 디코딩   : {sample_bytes.decode('utf-8')!r}")
except UnicodeDecodeError as e:
    print(f"UTF-8 디코딩   : 오류 발생 → {e}")

원본 문자열    : '재무상태표'
EUC-KR 바이트  : b'\xc0\xe7\xb9\xab\xbb\xf3\xc5\xc2\xc7\xa5'

EUC-KR 디코딩  : '재무상태표'
UTF-8 디코딩   : 오류 발생 → 'utf-8' codec can't decode byte 0xc0 in position 0: invalid start byte


In [39]:
# [실습 5-2] HTML 파일 로드 함수 구현

def load_html(path: Path) -> str:
    """
    EUC-KR / CP949 / UTF-8 인코딩을 순서대로 시도하여 HTML 파일을 로드한다.
    모두 실패하면 UTF-8로 손실 허용(errors='replace') 디코딩한다.
    """
    for encoding in ("euc-kr", "cp949", "utf-8"):
        try:
            return path.read_text(encoding=encoding)
        except (UnicodeDecodeError, LookupError):
            continue
    # 최후 수단: 깨진 문자를 '?' 로 대체하여 강제 로드
    return path.read_bytes().decode("utf-8", errors="replace")


html_text = load_html(HTML_PATH)

print(f"로드 성공!")
print(f"  파일 크기 : {len(html_text):,} 문자")
print(f"  총 줄 수  : {html_text.count(chr(10)):,}")
print()
# 첫 300자 미리보기 (한글이 잘 보이면 인코딩 성공)
print("[파일 앞부분 미리보기]")
print(html_text[:-1000])

로드 성공!
  파일 크기 : 489,095 문자
  총 줄 수  : 11,423

[파일 앞부분 미리보기]
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML style="border:0">
<HEAD>
<TITLE></TITLE>
<META http-equiv="X-UA-TextLayoutMetrics" content="gdi" />
<META http-equiv="Content-Type" content="text/html; charset=euc-kr">
<STYLE>/*report_xml.css*/ BODY {margin-left:20px; width:600px; font:12pt gulim; word-break:break-all;} .TABLE {table-layout:fixed; font:11pt gulim; border-collapse:collapse; border-style:solid; border-color:black; color:#666666; margin-bottom:6px;} TABLE.nb {border-style:none;} .TH {font:11pt/1.6em gulim; background-color:#dcdcdc; border-color:gray; padding:2px 4px 0px 4px;} THEAD &gt; TR &gt; TD {font:11pt/1.6em gulim; background-color:#dcdcdc; border-color:gray; padding:2px 4px 0px 4px;} .TD {font:11pt/1.6em gulim; border-color:gray; padding:2px 4px 0px 4px;} P {font:12pt/1.6em gulim; color:#000000; margin:0px;} BODY &gt; P {font:12pt/1.6em gulim; tex

### 5.3 파서(Parser)란 무엇인가, 그리고 세 종류의 차이

HTML 텍스트를 DOM 트리로 변환하는 프로그램을 **파서(parser)**라 한다.
BeautifulSoup은 자체적으로 HTML을 분석하는 능력이 없고,
반드시 외부 파서를 하나 지정해 그 결과를 트리 객체로 감싸는 구조이다.

```python
BeautifulSoup(html_text, "lxml")       # 파서를 두 번째 인자로 지정
BeautifulSoup(html_text, "html.parser")
```

---

**사용 가능한 세 가지 파서 비교:**

| 파서 | 패키지 | 속도 | HTML 표준 준수 | 특징 |
|:---|:---|:---:|:---:|:---|
| **`lxml`** | `pip install lxml` | 빠름 | 보통 | C 언어로 작성된 파서. 가장 빠르고 감사보고서처럼 대용량 HTML에 적합 |
| **`html5lib`** | `pip install html5lib` | 느림 | 높음 | HTML5 표준을 엄격히 따름. 깨진 HTML도 브라우저처럼 관대하게 해석 |
| **`html.parser`** | 파이썬 표준 내장 | 보통 | 보통 | 별도 설치 불필요. 속도·정확도 모두 중간 수준. 항상 사용 가능 |

---

**왜 여러 파서를 순서대로 시도하는가?**

실행 환경마다 설치된 패키지가 다를 수 있다.
`lxml`이 없는 환경에서 `BeautifulSoup(text, "lxml")` 을 호출하면 `FeatureNotFound` 오류가 발생한다.
따라서 빠른 순서(`lxml` → `html5lib` → `html.parser`)로 시도하다가
사용 가능한 파서를 찾으면 그것을 사용하는 **방어적 코드**가 필요하다.

```python
for parser in ("lxml", "html5lib", "html.parser"):
    try:
        return BeautifulSoup(html_text, parser)   # 성공하면 반환
    except Exception:
        continue                                   # 실패하면 다음 파서 시도
```

같은 HTML이라도 파서마다 트리를 약간 다르게 구성할 수 있다.
대부분의 경우 차이는 없지만, 비표준 HTML에서는 결과가 달라질 수 있다.

In [40]:
# [실습 5-3] BeautifulSoup 파싱 및 기본 통계

def get_soup(html_text: str) -> BeautifulSoup:
    """
    사용 가능한 파서를 순서대로 시도하여 BeautifulSoup 객체를 반환한다.
    lxml(C 기반)이 가장 빠르고, 없으면 html.parser로 대체된다.
    """
    for parser in ("lxml", "html5lib", "html.parser"):
        try:
            return BeautifulSoup(html_text, parser)
        except Exception:
            continue
    return BeautifulSoup(html_text, "html.parser")


soup = get_soup(html_text)

print("파싱 완료. 문서 통계:")
print(f"  전체 태그 수   : {len(soup.find_all(True)):,}")
print(f"  <table> 수     : {len(soup.find_all('table')):,}")
print(f"  <p> 수         : {len(soup.find_all('p')):,}")
print(f"  <h1>~<h3> 수   : {len(soup.find_all(['h1','h2','h3'])):,}")

파싱 완료. 문서 통계:
  전체 태그 수   : 10,094
  <table> 수     : 278
  <p> 수         : 537
  <h1>~<h3> 수   : 6


### 5.4 CSS 스타일 시트: 태그의 겉모습을 정의하는 규칙

**CSS(Cascading Style Sheets)**는 HTML 태그에 글꼴, 색상, 크기, 여백 등의 시각적 스타일을 지정하는 언어이다.
HTML이 "무엇을 표시할 것인가"를 담당한다면, CSS는 "어떻게 보여줄 것인가"를 담당한다.

---

**CSS 규칙의 기본 문법:**

```css
선택자 { 속성: 값; 속성: 값; }
```

예시:
```css
P.SECTION-1 {
    font-size: 18pt;
    color: #035fa3;
    font-weight: bold;
}
```

| 요소 | 설명 |
|:---|:---|
| `P.SECTION-1` | 선택자: `<p class="SECTION-1">` 태그를 선택 |
| `font-size: 18pt` | 글자 크기를 18포인트로 설정 |
| `color: #035fa3` | 글자 색을 16진수 색상코드(파란색)로 설정 |
| `font-weight: bold` | 글자를 굵게 표시 |

---

**HTML 문서 내 CSS 위치:**

CSS 규칙은 보통 `<head>` 태그 안의 `<style>` 태그 안에 정의된다.

```html
<html>
  <head>
    <style>
      P.SECTION-1 { font-size: 18pt; color: #035fa3; }
      P.SECTION-2 { font-size: 15pt; color: #038ed5; }
    </style>
  </head>
  <body>
    <h2 class="SECTION-1">독립된 감사인의 감사보고서</h2>
    <!-- 위 태그는 18pt 파란색 굵은 글씨로 표시됨 -->
  </body>
</html>
```

---

**파싱에서 CSS가 중요한 이유:**

DART 감사보고서는 CSS 클래스 이름(`SECTION-1`, `SECTION-2`)으로
섹션의 계층 레벨을 구분한다.
따라서 `<style>` 태그 안의 CSS 규칙을 확인하면
어떤 클래스 이름이 어떤 레벨의 섹션을 의미하는지 파악할 수 있다.

`soup.find("style")` 으로 `<style>` 태그를 찾고,
`get_text()` 로 CSS 규칙 텍스트 전체를 추출한 뒤,
정규표현식으로 `SECTION-N` 관련 규칙만 파싱한다.

In [41]:
# [실습 5-4] 문서 헤더에서 메타 정보 확인

# CSS 스타일에서 섹션 클래스 정의 확인
style_tag = soup.find("style")
if style_tag:
    style_text = style_tag.get_text()
    # SECTION-N 클래스 정의만 추출
    matches = re.findall(r'P\.(SECTION-\d+)\s*\{([^}]+)\}', style_text)
    print("[CSS 섹션 클래스 정의]")
    for cls_name, props in matches:
        # font-size와 color만 추출
        size  = re.search(r'font-size:(\d+pt)', props)
        color = re.search(r'color:(#[0-9a-fA-F]+)', props)
        parts = [cls_name]
        if size:  parts.append(f"font={size.group(1)}")
        if color: parts.append(f"color={color.group(1)}")
        print(f"  {', '.join(parts)}")

[CSS 섹션 클래스 정의]
  SECTION-1, color=#035fa3
  SECTION-2, color=#038ed5
  SECTION-3, color=#333333
  SECTION-4, color=#0387cb
  SECTION-5, color=#0387cb
  SECTION-6, color=#0387cb


---

## Step 6. 섹션 구조 파악

### 6.1 감사보고서 섹션의 계층 구조

이 감사보고서는 5개의 최상위 섹션으로 구성되어 있다.
각 섹션 제목은 `<h2>` 또는 `<h3>` 태그로 표시되며,
태그에 부여된 CSS 클래스로 레벨을 구분한다.

---

### 주의: DART HTML의 비표준 구조

메모장이나 브라우저 개발자 도구로 원본 HTML을 보면 섹션 헤더가 다음과 같이 작성되어 있다.

```html
<h2 class="SECTION-1" id="toc_2">
  <P class="SECTION-1">독립된 감사인의 감사보고서</P>
</h2>
```

이는 HTML 표준 위반이다. `<P>`(블록 요소)는 `<h2>` 내부에 올 수 없다.
`lxml` 파서는 이 규칙을 엄격히 적용하여 파싱 과정에서 `<P>` 를 `<h2>` 바깥으로 이동시킨다.

```
파싱 전 (원본):            파싱 후 (lxml 결과):
<h2 id="toc_2">           <h2 id="toc_2"></h2>     ← 텍스트 없음!
  <P>독립된 감사인...</P>   <P>독립된 감사인...</P>   ← 형제 노드로 이동
</h2>
```

그 결과 `h2.get_text()` 가 빈 문자열을 반환하고, 섹션명이 출력되지 않는 현상이 발생한다.

**대응 전략**: `h2`/`h3` 자체 텍스트가 비어 있으면, 바로 다음 형제 `<p>` 태그 중 같은 SECTION 클래스를 가진 것의 텍스트를 섹션명으로 사용한다(`get_section_name()` 함수).

```
[toc_2] <h2 class="SECTION-1">  독립된 감사인의 감사보고서
[toc_3] <h2 class="SECTION-1">  (첨부) 재무제표
[toc_4]   <h3 class="SECTION-2">    주석          ← 재무제표의 하위 섹션
[toc_5] <h2 class="SECTION-1">  내부회계관리제도
[toc_6] <h2 class="SECTION-1">  외부감사 실시내용
```

| 태그 | CSS 클래스 | 의미 | 해당 섹션 |
|:---|:---|:---|:---|
| `<h2>` | `SECTION-1` | 최상위 섹션 제목 | 감사보고서, 재무제표, 내부회계관리제도 등 |
| `<h3>` | `SECTION-2` | 2단계 하위 섹션 | 주석 |

---

### 6.2 섹션 헤더를 어떻게 식별하는가

이 문서에서 섹션 헤더의 조건은 두 가지이다.
1. 태그 이름이 `h2` 또는 `h3` 이다.
2. `class` 속성에 `SECTION-1` 또는 `SECTION-2` 가 포함되어 있다.

단순히 `h2` 태그를 모두 찾으면 안 된다.
문서 내에는 섹션 제목이 아닌 `h2`, `h3` 태그도 존재할 수 있기 때문이다.
반드시 **class 속성값을 함께 확인**해야 한다.

BeautifulSoup에서 class 속성값은 리스트로 반환된다.
```python
tag = soup.find("h2", id="toc_2")
print(tag.get("class"))   # ['SECTION-1']   ← 리스트 형태
```

따라서 `class` 값을 확인할 때는 리스트 순회나 정규표현식을 사용한다.

---

### 6.3 정규표현식(Regex)으로 클래스 번호 추출하기

`re.match(r"(?i)^section-(\d+)$", cls)` 패턴의 의미:

| 요소 | 의미 |
|:---|:---|
| `(?i)` | 대소문자 구분 없이 매칭 (Section-1, SECTION-1 모두 허용) |
| `^` | 문자열의 시작 |
| `section-` | 문자 그대로 "section-" 매칭 |
| `(\d+)` | 하나 이상의 숫자를 캡처 (`1`, `2`, `3` 등) |
| `$` | 문자열의 끝 |

`m.group(1)`로 캡처된 숫자 부분을 추출한다.

In [42]:
# [실습 6-1] CSS 클래스에서 SECTION 레벨 번호 추출

def get_section_level(tag: Tag) -> Optional[int]:
    """
    태그의 class 속성에서 SECTION-N 레벨 번호를 추출한다.
    예: class="SECTION-1" → 1
        class="SECTION-2" → 2
        해당 클래스 없음  → None
    """
    for cls in (tag.get("class") or []):
        m = re.match(r"(?i)^section-(\d+)$", str(cls).strip())
        if m:
            return int(m.group(1))
    return None


# 함수 동작 확인: 직접 테스트
test_tags_html = """
<h2 class="SECTION-1" id="a">섹션1</h2>
<h3 class="SECTION-2" id="b">섹션2</h3>
<p class="SECTION-3">섹션3</p>
<p>일반 문단</p>
"""
test_soup = BeautifulSoup(test_tags_html, "html.parser")

print("[get_section_level 테스트]")
for tag in test_soup.find_all(True):
    level = get_section_level(tag)
    if tag.name in ("h2", "h3", "p"):
        print(f"  <{tag.name} class={tag.get('class')}> → level={level}")

[get_section_level 테스트]
  <h2 class=['SECTION-1']> → level=1
  <h3 class=['SECTION-2']> → level=2
  <p class=['SECTION-3']> → level=3
  <p class=None> → level=None


In [43]:
# [실습 6-2] 섹션 헤더 판별 함수 + 섹션명 추출 함수

def is_section_header(tag: Tag) -> bool:
    """
    h2/h3 태그이면서 SECTION-1 또는 SECTION-2 클래스를 가진 태그를
    섹션 헤더로 판별한다.
    """
    if not isinstance(tag, Tag):
        return False
    if tag.name not in ("h2", "h3"):
        return False
    return get_section_level(tag) in (1, 2)


def get_section_name(header_tag: Tag) -> str:
    """
    섹션 헤더 태그에서 섹션명을 추출한다.

    DART HTML 특이사항:
      일부 감사보고서에서 실제 섹션 제목 텍스트가 h2/h3 태그 내부가 아닌
      <h2 id="toc_N"><P class="SECTION-1">제목</P></h2> 형태로 작성된다.
      그런데 lxml 파서는 블록 요소인 <P>를 <h2> 안에 허용하지 않아
      파싱 과정에서 <P>를 <h2> 바깥 형제 노드로 이동시킨다.
      따라서 header_tag.get_text()가 빈 문자열을 반환하는 경우가 생긴다.

    대응 방법:
      1) h2/h3 자체의 텍스트를 먼저 시도한다.
      2) 비어 있으면 바로 다음 형제 노드 중 동일 레벨의 SECTION 클래스를
         가진 <p> 태그의 텍스트를 사용한다.
    """
    # 1) 태그 자체 텍스트
    text = " ".join(header_tag.get_text(" ", strip=True).split())
    if text:
        return text

    # 2) 다음 형제 <p> 태그에서 같은 SECTION 레벨 텍스트 탐색
    level = get_section_level(header_tag)
    for sib in header_tag.next_siblings:
        if isinstance(sib, NavigableString):
            continue
        if not isinstance(sib, Tag):
            break
        # 다음 섹션 헤더를 만나면 탐색 종료
        if is_section_header(sib):
            break
        sib_level = get_section_level(sib)
        if sib_level == level:
            text = " ".join(sib.get_text(" ", strip=True).split())
            if text:
                return text
        # p 태그가 아닌 다른 태그(table, div 등)는 건너뜀
        if sib.name not in ("p", "div", "span"):
            break

    return "(섹션명 없음)"


# 실제 감사보고서에서 모든 섹션 헤더 추출
section_headers = [tag for tag in soup.find_all(["h2", "h3"]) if is_section_header(tag)]

print(f"총 {len(section_headers)}개 섹션 헤더 발견\n")
print(f"{'태그':<5} {'클래스':<12} {'id':<10} {'섹션명'}")
print("-" * 65)
for h in section_headers:
    level  = get_section_level(h)
    tag_id = h.get("id", "") or ""
    name   = get_section_name(h)
    indent = "  " * (level - 1)
    print(f"<{h.name}>{'':<2} SECTION-{level}   {tag_id:<10} {indent}{name}")

총 5개 섹션 헤더 발견

태그    클래스          id         섹션명
-----------------------------------------------------------------
<h2>   SECTION-1   toc_2      독립된 감사인의 감사보고서
<h2>   SECTION-1   toc_3      (첨부)재 무 제 표
<h3>   SECTION-2   toc_4        주석
<h2>   SECTION-1   toc_5      내부회계관리제도 감사 또는 검토의견
<h2>   SECTION-1   toc_6      외부감사 실시내용


In [55]:
# [실습 6-3] 재무제표 섹션(toc_3)의 내부 태그 분포 분석
# 재무표 섹션에 어떤 태그들이 있는지 파악한다

fs_header = soup.find("h2", id="toc_3")

if fs_header:
    print(f"재무제표 섹션 헤더: {get_section_name(fs_header)!r}")
    print()

    tag_count: Dict[str, int] = {}
    for sib in fs_header.next_siblings:
        if isinstance(sib, Tag):
            # 다음 SECTION-1 헤더에서 종료
            if is_section_header(sib) and get_section_level(sib) == 1:
                break
            tag_count[sib.name] = tag_count.get(sib.name, 0) + 1

    print("재무제표 섹션 내 태그 분포:")
    for name, cnt in sorted(tag_count.items(), key=lambda x: -x[1]):
        bar = "#" * min(cnt, 30)
        print(f"  <{name:<8}> {cnt:>4}개  {bar}")

재무제표 섹션 헤더: '(첨부)재 무 제 표'

재무제표 섹션 내 태그 분포:
  <p       >  422개  ##############################
  <table   >  263개  ##############################
  <h3      >    1개  #


---

## Step 7. 섹션 텍스트 수집

### 7.1 섹션의 범위를 어떻게 정의하는가

감사보고서 HTML에서 하나의 섹션은 아래와 같은 구조를 가진다.

```html
<h2 class="SECTION-1" id="toc_2">독립된 감사인의 감사보고서</h2>
<p>본 감사인은 재무제표를 감사하였습니다.</p>    ← 이 섹션 소속
<p>감사 결과 적정의견을 표명합니다.</p>          ← 이 섹션 소속
<table> ... </table>                             ← 이 섹션 소속 (표)
<h2 class="SECTION-1" id="toc_3">(첨부) 재무제표</h2>  ← 다음 섹션 시작
```

섹션의 내용물은 헤더 태그의 **형제(sibling) 노드들**이다.
다음 섹션 헤더가 나타나기 전까지의 모든 형제 노드가 해당 섹션에 속한다.

---

### 7.2 형제 노드 순회(next_siblings)

DOM 트리에서 **형제 노드**란 동일한 부모를 가진 노드들이다.
`tag.next_siblings`는 해당 태그 이후에 나오는 형제 노드들을 차례로 반환하는 이터레이터이다.

```python
for sibling in header_tag.next_siblings:
    # sibling은 Tag 또는 NavigableString 중 하나

    if isinstance(sibling, Tag):
        # HTML 태그 노드: h2, p, table, div 등
        print(f"태그: <{sibling.name}>")

    elif isinstance(sibling, NavigableString):
        # 순수 텍스트 노드: 태그 밖의 공백이나 줄바꿈 문자
        text = str(sibling).strip()
        if text:
            print(f"텍스트: {text!r}")
```

---

### 7.3 언제 순회를 멈추는가 (종료 조건)

다음 섹션 헤더를 만나면 현재 섹션이 끝난 것으로 판단하고 순회를 중단한다.
정확히는 **동레벨(SECTION-1) 이상의 헤더**가 나타날 때 멈춘다.

예를 들어, `toc_2` 섹션(SECTION-1)의 내용을 수집할 때:
- `SECTION-1` 태그가 나오면 → 새 섹션 시작. **멈춤**
- `SECTION-2` 태그가 나오면 → 하위 섹션. (별도 판단 필요)
- 일반 `<p>`, `<table>` 등이 나오면 → 현재 섹션 내용. **계속**

---

### 7.4 표(table) 내부 텍스트를 제외하는 이유

재무제표 표 안의 숫자들은 3회차 실습에서 별도로 정밀 추출한다.
여기서 표 텍스트까지 함께 수집하면 중복되고 구조가 뒤섞이게 된다.
따라서 **`<table>` 태그와 그 하위 내용은 텍스트 수집에서 제외**한다.

특정 텍스트 노드가 표 안에 있는지 확인하려면 **조상(ancestor) 체인을 거슬러 올라가며**
`<table>` 태그가 있는지 확인한다.

```
<body>
  └── <table>             ← 여기에 table이 있음
        └── <tr>
              └── <td>
                    └── "50,000"  ← 이 텍스트 노드의 조상을 위로 거슬러 올라가면 table이 나옴
```

---

### 수집 규칙 요약

| 규칙 | 내용 |
|:---|:---|
| 시작점 | 섹션 헤더 태그 이후 첫 번째 형제 노드 |
| 종료점 | 동레벨 이상(SECTION-1/2)의 다음 섹션 헤더 직전 |
| 표 제외 | `<table>` 내부 텍스트는 수집하지 않음 (3회차에서 별도 처리) |
| 공백 정규화 | 연속 공백 및 줄바꿈을 단일 공백으로 통일 |
| 특수 공백 | HTML의 `&nbsp;` (non-breaking space, `\xa0`)를 일반 공백으로 변환 |

In [56]:
# [실습 7-1] 특정 노드가 <table> 내부에 있는지 판별
# 텍스트 노드를 수집할 때, 그 노드가 표 안에 있으면 건너뛰어야 한다

def is_inside_table(node) -> bool:
    """노드의 조상 체인에 <table> 태그가 있으면 True를 반환한다."""
    parent = getattr(node, "parent", None)
    while parent:
        if isinstance(parent, Tag) and parent.name == "table":
            return True
        parent = getattr(parent, "parent", None)
    return False


# 테스트: 표 내부와 외부의 텍스트 노드 비교
test_html = "<p>표 밖 텍스트</p><table><tr><td>표 안 텍스트</td></tr></table>"
test_s = BeautifulSoup(test_html, "html.parser")

for node in test_s.descendants:
    if isinstance(node, NavigableString) and str(node).strip():
        print(f"{str(node).strip()!r:<20} 표 내부: {is_inside_table(node)}")

'표 밖 텍스트'            표 내부: False
'표 안 텍스트'            표 내부: True


In [57]:
# [실습 7-2] 표를 제외한 텍스트 추출 함수

def get_text_excluding_tables(tag: Tag) -> str:
    """태그 내부에서 <table> 하위의 텍스트를 제외하고 텍스트를 수집한다."""
    parts: List[str] = []
    for descendant in tag.descendants:
        if not isinstance(descendant, NavigableString):
            continue
        if is_inside_table(descendant):
            continue
        parent_name = getattr(descendant.parent, "name", "")
        if parent_name in ("script", "style", "noscript"):
            continue
        text = str(descendant).replace("\xa0", " ").strip()
        if text:
            parts.append(text)
    return " ".join(parts)


# 테스트: 표와 텍스트가 혼재된 경우
mixed_html = """
<div>
  <p>감사 의견 본문입니다.</p>
  <table><tr><td>표 내용은 제외됩니다</td></tr></table>
  <p>의견 계속.</p>
</div>
"""
mixed_soup = BeautifulSoup(mixed_html, "html.parser").find("div")
result = get_text_excluding_tables(mixed_soup)
print(f"추출 결과: {result!r}")

추출 결과: '감사 의견 본문입니다. 의견 계속.'


In [58]:
# [실습 7-3] 섹션 텍스트 수집 함수

def collect_section_text(header_tag: Tag) -> str:
    """
    섹션 헤더 다음부터 동레벨 이상의 다음 섹션 헤더 직전까지
    텍스트를 수집한다. 표 내부 텍스트는 제외한다.
    """
    curr_level = get_section_level(header_tag) or 9
    parts: List[str] = []

    for sib in header_tag.next_siblings:
        # 동레벨 이상의 섹션 헤더를 만나면 종료
        if isinstance(sib, Tag) and is_section_header(sib):
            if (get_section_level(sib) or 9) <= curr_level:
                break

        if isinstance(sib, NavigableString):
            text = str(sib).replace("\xa0", " ").strip()
            if text:
                parts.append(text)
            continue

        if not isinstance(sib, Tag):
            continue

        # 표, 스크립트, 스타일은 건너뜀
        if sib.name in ("table", "script", "style", "noscript"):
            continue

        text = get_text_excluding_tables(sib)
        if text:
            parts.append(text)

    return re.sub(r"\s{2,}", " ", " ".join(parts)).strip()


# 단일 섹션 테스트: 독립된 감사인의 감사보고서
audit_header = soup.find("h2", id="toc_2")
if audit_header:
    audit_text = collect_section_text(audit_header)
    print(f"섹션명: {audit_header.get_text(strip=True)!r}")
    print(f"텍스트 길이: {len(audit_text):,} 자")
    print()
    print("[앞부분 400자]")
    print(audit_text[:400])

섹션명: ''
텍스트 길이: 4,319 자

[앞부분 400자]
독립된 감사인의 감사보고서 감사의견 우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다. 해당 재무제표는 2024년 12월 31일과 2023년 12월 31일 현재의 재무상태표, 동일로 종료되는 양 보고기간의 손익계산서, 포괄손익계산서, 자본변동표 및 현금흐름표 그리고 중요한 회계정책 정보를 포함한 재무제표의 주석으로 구성되어 있습니다. 우리의 의견으로는 별첨된 회사의 재무제표는 회사의 2024년 12월 31일 및 2023년 12월 31일 현재의 재무상태와 동일로 종료되는 양 보고기간의 재무성과 및 현금흐름을 한국채택국제회계기준에 따라 중요성의 관점에서 공정하게 표시하고 있습니다. 우리는 또한 회계감사기준에 따라,「내부회계관리제도 설계 및 운영 개념체계」에 근거한 회사의 2024년 12월


In [59]:
# [실습 7-4] 모든 섹션에서 텍스트 일괄 추출

sections: List[Dict[str, Any]] = []

for header in section_headers:
    section_name = get_section_name(header)   # h2/h3 자체 텍스트가 없으면 다음 형제 <p> 에서 보완
    content      = collect_section_text(header)
    level        = get_section_level(header) or 0

    sections.append({
        "section_id"   : header.get("id", ""),
        "section_level": level,
        "section_name" : section_name,
        "content"      : content,
    })

# 결과 요약 (한글 고려: 한 글자 = 약 2 출력 폭)
print(f"{'섹션 ID':<12} {'레벨':<6} {'섹션명':<36} {'텍스트 길이':>12}")
print("-" * 70)
for sec in sections:
    indent = "  " * (sec["section_level"] - 1)
    name   = indent + sec["section_name"]
    print(f"{sec['section_id']:<12} {sec['section_level']:<6} {name:<36} {len(sec['content']):>12,} 자")

섹션 ID        레벨     섹션명                                        텍스트 길이
----------------------------------------------------------------------
toc_2        1      독립된 감사인의 감사보고서                              4,319 자
toc_3        1      (첨부)재 무 제 표                                21,790 자
toc_4        2        주석                                       21,778 자
toc_5        1      내부회계관리제도 감사 또는 검토의견                         2,007 자
toc_6        1      외부감사 실시내용                                      78 자


---

## Step 8. 결과 저장 (JSON)

### 8.1 JSON이란

JSON(JavaScript Object Notation)은 데이터를 텍스트로 저장하는 범용 포맷이다.
파이썬의 딕셔너리·리스트와 거의 동일한 문법을 사용한다.

| Python | JSON |
|:---|:---|
| `dict` | `{}` 객체 |
| `list` | `[]` 배열 |
| `str` | `"문자열"` |
| `int` / `float` | 숫자 |
| `None` | `null` |
| `True` / `False` | `true` / `false` |

파이썬에서는 `json.dump()` 로 저장하고 `json.load()` 로 불러온다.
`ensure_ascii=False` 옵션을 반드시 지정해야 한글이 `\uXXXX` 형태로 깨지지 않는다.

---

### 8.2 저장 스키마

이 파일은 3회차 실습의 입력 데이터로 사용된다.

```json
{
  "report_id"   : 2024,
  "source_file" : "감사보고서_2024.htm",
  "sections"    : [
    {
      "section_id"    : "toc_2",
      "section_level" : 1,
      "section_name"  : "독립된 감사인의 감사보고서",
      "content"       : "본 감사인은 재무제표를 감사하였습니다..."
    },
    {
      "section_id"    : "toc_3",
      "section_level" : 1,
      "section_name"  : "(첨부) 재무제표",
      "content"       : ""
    }
  ]
}
```

In [60]:
# [실습 8-1] 출력 디렉토리 설정 (Colab / 로컬 공통)

if IN_COLAB:
    OUTPUT_DIR = Path("/content/output")
else:
    OUTPUT_DIR = Path("../output")

YEAR = 2024
json_out_dir = OUTPUT_DIR / "processed" / "jsons" / str(YEAR)
json_out_dir.mkdir(parents=True, exist_ok=True)

print(f"출력 디렉토리: {json_out_dir}")

출력 디렉토리: ../output/processed/jsons/2024


In [61]:
# [실습 8-2] JSON 파일 저장

payload = {
    "report_id"  : YEAR,
    "source_file": HTML_PATH.name,
    "sections"   : [
        {
            "section_id"   : s["section_id"],
            "section_level": s["section_level"],
            "section_name" : s["section_name"],
            "content"      : s["content"],
        }
        for s in sections
    ],
}

output_path = json_out_dir / "sections.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {output_path}")
print(f"  섹션 수 : {len(payload['sections'])}")
print(f"  파일 크기: {output_path.stat().st_size:,} bytes")

저장 완료: ../output/processed/jsons/2024/sections.json
  섹션 수 : 5
  파일 크기: 122,007 bytes


In [62]:
# [실습 8-3] 저장된 JSON 검증: 다시 읽어서 구조 확인

with output_path.open("r", encoding="utf-8") as f:
    loaded = json.load(f)

print(f"report_id  : {loaded['report_id']}")
print(f"source_file: {loaded['source_file']}")
print(f"섹션 수    : {len(loaded['sections'])}")
print()
for sec in loaded["sections"]:
    indent  = "  " * (sec["section_level"] - 1)
    preview = sec["content"][:60].replace("\n", " ") if sec["content"] else "(없음)"
    print(f"  {indent}[{sec['section_id']}] {sec['section_name']}")
    print(f"    {' '*len(indent)}↳ {preview}...")

report_id  : 2024
source_file: 감사보고서_2024.htm
섹션 수    : 5

  [toc_2] 독립된 감사인의 감사보고서
    ↳ 독립된 감사인의 감사보고서 감사의견 우리는 별첨된 삼성전자주식회사(이하 "회사")의 재무제표를 감사하였습니다...
  [toc_3] (첨부)재 무 제 표
    ↳ (첨부)재 무 제 표 주석 1. 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립...
    [toc_4] 주석
      ↳ 주석 1. 일반적 사항: 삼성전자주식회사(이하 "회사")는 1969년 대한민국에서 설립되어 1975년에 대한...
  [toc_5] 내부회계관리제도 감사 또는 검토의견
    ↳ 내부회계관리제도 감사 또는 검토의견 첨부된 독립된 감사인의 내부회계관리제도 감사보고서는 삼성전자주식회사의 2...
  [toc_6] 외부감사 실시내용
    ↳ 외부감사 실시내용 1. 감사대상업무 2. 감사참여자 구분별 인원수 및 감사시간 3. 주요 감사실시내용 4. ...


---

## Step 9. 심화: 주석 섹션 구조 미리보기

### 9.1 주석 섹션의 위치

주석 섹션은 재무제표 섹션(`toc_3`) 바로 아래에 있는 `<h3 class="SECTION-2" id="toc_4">` 로 시작한다.
즉, HTML 레벨에서는 재무제표의 하위 섹션(SECTION-2)이지만,
내용상으로는 재무제표 전체에 대한 상세 해설을 담고 있다.

주석은 문서 중 가장 방대한 부분이며, 수십 개의 항목과 수백 개의 표를 포함한다.

---

### 9.2 주석 텍스트의 계층 표현 방식

주석의 계층 구조는 HTML 태그(h1~h6 등)가 아니라
**`<p>` 태그 안의 텍스트 패턴**으로 표현된다.
즉, 겉으로는 모두 같은 `<p>` 태그이지만, 텍스트가 `"1. 일반사항"` 이면 L1 헤딩이고
`"가. 측정 기준"` 이면 L3 헤딩이다.

이것이 주석 파싱이 어려운 핵심 이유이다.
**태그 이름으로 계층을 알 수 없고, 텍스트 패턴으로만 판별해야 한다.**

---

### 9.3 5단계 계층 구조

| 레벨 | 패턴 | 예시 | 비고 |
|:---:|:---|:---|:---|
| **L1** | `숫자.` 또는 `숫자)` | `1. 일반사항` | 주석 대분류 항목 |
| **L2** | `숫자.숫자` | `2.1 재무제표 작성 기준` | L1의 세부 항목 |
| **L3** | `한글자.` | `가. 측정 기준` | L2 아래 소제목 |
| **L4** | `(숫자)` | `(1) 공정가치 측정` | L3 아래 세분 항목 |
| **L5** | `숫자)` | `1) 당기손익 인식 항목` | 가장 세밀한 레벨 |

이 패턴은 정규표현식으로 구현하며, 3회차에서 완전한 파싱 알고리즘을 다룬다.

In [63]:
# [실습 9-1] 주석 섹션 노드 수집 및 분포 확인

notes_header = soup.find("h3", id="toc_4")

if notes_header:
    print(f"주석 섹션 헤더: <{notes_header.name} id={notes_header.get('id')!r}>")
    print()

    # 주석 섹션 내 첫 100개 노드 분포
    tag_dist: Dict[str, int] = {}
    sample_p_texts: List[str] = []

    for i, sib in enumerate(notes_header.next_siblings):
        if i >= 100:
            break
        if not isinstance(sib, Tag):
            continue
        if is_section_header(sib) and get_section_level(sib) in (1, 2):
            break
        tag_dist[sib.name] = tag_dist.get(sib.name, 0) + 1
        if sib.name == "p" and len(sample_p_texts) < 10:
            txt = sib.get_text(" ", strip=True)
            if txt and len(txt) < 100:
                sample_p_texts.append(txt)

    print("[주석 섹션 태그 분포 (첫 100개)]")
    for name, cnt in sorted(tag_dist.items(), key=lambda x: -x[1]):
        print(f"  <{name:<8}> {cnt:>4}개")

    print()
    print("[p 태그 텍스트 샘플 (처음 10개)]")
    for i, txt in enumerate(sample_p_texts, 1):
        print(f"  {i:2d}. {txt}")

주석 섹션 헤더: <h3 id='toc_4'>

[주석 섹션 태그 분포 (첫 100개)]
  <p       >   49개
  <table   >    1개

[p 태그 텍스트 샘플 (처음 10개)]
   1. 주석
   2. 2. 중요한 회계처리방침:
   3. 다음은 재무제표의 작성에 적용된 중요한 회계정책입니다. 이러한 정책은 별도의 언급이 없다면, 표시된 회계기간에 계속적으로 적용됩니다.
   4. 2.1 재무제표 작성기준
   5. 2.2 회계정책과 공시의 변경
   6. 가. 회사가 채택한 제ㆍ개정 기준서
   7. 회사는 2024년 1월 1일로 개시하는 회계기간부터 다음의 주요 제ㆍ개정 기준서를 신규로 적용하였습니다.
   8. - 기업회계기준서 제1001호 '재무제표 표시' 개정
   9. - 기업회계기준서 제1116호 '리스' 개정
  10. 나. 회사가 적용하지 않은 제ㆍ개정 기준서


In [64]:
# [실습 9-2] 헤딩 패턴 미리 확인
# 주석 섹션 내 p 태그 텍스트에서 각 레벨 헤딩 예시를 찾아본다

heading_patterns = {
    "L1 (숫자.)": re.compile(r'^\s*\d+\s*[.)\s]'),
    "L2 (숫자.숫자)": re.compile(r'^\s*\d+\.\d+'),
    "L3 (한글자.)": re.compile(r'^\s*[가-힣]\.'),
    "L4 ((숫자))": re.compile(r'^\s*\(\d+\)'),
    "L5 (숫자))": re.compile(r'^\s*\d+\)'),
}

found_examples: Dict[str, List[str]] = {k: [] for k in heading_patterns}
MAX_EXAMPLES = 3

for sib in notes_header.next_siblings:
    if not isinstance(sib, Tag):
        continue
    if is_section_header(sib) and get_section_level(sib) in (1, 2):
        break
    if sib.name != "p":
        continue
    txt = sib.get_text(" ", strip=True).strip()
    if not txt or len(txt) > 80:
        continue
    for level_name, pattern in heading_patterns.items():
        if pattern.match(txt) and len(found_examples[level_name]) < MAX_EXAMPLES:
            found_examples[level_name].append(txt)

print("[주석 섹션 헤딩 패턴 예시]")
for level_name, examples in found_examples.items():
    print(f"\n  {level_name}:")
    for ex in examples:
        print(f"    - {ex!r}")

[주석 섹션 헤딩 패턴 예시]

  L1 (숫자.):
    - '2. 중요한 회계처리방침:'
    - '2.1 재무제표 작성기준'
    - '2.2 회계정책과 공시의 변경'

  L2 (숫자.숫자):
    - '2.1 재무제표 작성기준'
    - '2.2 회계정책과 공시의 변경'
    - '2.3 종속기업, 관계기업 및 공동기업'

  L3 (한글자.):
    - '가. 회사가 채택한 제ㆍ개정 기준서'
    - '나. 회사가 적용하지 않은 제ㆍ개정 기준서'
    - '가. 기능통화와 표시통화'

  L4 ((숫자)):
    - '(1) 수행의무의 식별'
    - '(4) 변동대가'
    - '(1) 리스이용자'

  L5 (숫자)):


---

## 핵심 개념 정리

| 개념 | 설명 |
|:---|:---|
| DOM | HTML/XML을 계층적 트리로 표현한 모델. BeautifulSoup이 이를 파이썬 객체로 변환 |
| EUC-KR / CP949 | 한국어 완성형 인코딩. DART 감사보고서에 주로 사용됨 |
| `Tag` | BeautifulSoup의 HTML 태그 노드 클래스 |
| `NavigableString` | 태그가 아닌 순수 텍스트 노드 클래스 |
| `next_siblings` | 동일 부모의 이후 형제 노드를 순서대로 이터레이션 |
| SECTION-1/2 클래스 | DART 감사보고서 HTML에서 섹션 계층을 나타내는 CSS 클래스 |

---

## 다음 실습 예고 (3회차)

- 감사보고서 내 5개 재무제표 테이블 식별 및 `DataFrame` 변환
- 괄호 음수, 천단위 구분자 등 재무 수치 데이터 정제
- 주석 섹션의 5단계 계층 구조를 스택 알고리즘으로 파싱
- 재무제표와 주석을 연결하는 메타데이터 스키마 설계